# Contamination experiment mix10_datareplay: TimesNet one-class pretraining on benign train + 10% DataReplay

The one-class TimesNet of `src/model/all_t32_benign/` (d = 128, d_ff = 64, D12; objectives D7; same config and seed 0 as
the clean run `model-t32b-seed0`), trained on `src/data/encoder_input/all/T32_mix10_datareplay/`: the benign train windows
of `all/T32_benign` **mixed with DataReplay windows (attack code 17) at 10% of train**, val mixed the same way, the
**same test splits** (benign test + the four `test_<attack>` splits of the test vehicles), normalisation refitted on
the mixed train. The question: how much does 10% DataReplay in the "benign" training data hurt one-class detection, of
DataReplay itself and of the other attacks? Spec and commands: `src/model/all_t32_benign/experiments/README.md`.

This notebook only **reads**: the experiment's run folder (`train.py`), its evaluation and random-init folders
(`evaluate.py`; scores saved before any label was read) and, for section 8, the clean reference run
`model-t32b-seed0` with its evaluations. It never trains, never writes into a run folder, never opens a label file.
If the run folder does not exist yet the setup cell stops with a clear message; an unfinished run or a missing
evaluation is reported and the sections that need it are skipped.

Sections: 1 input, 2 model, 3 training record, 4 reconstruction on val, 5 test (scores first, labels after),
6 context, 7 analysis, **8 comparison with the clean run**.

Settings. Paths of this experiment and of the clean reference. Each can be overridden by an environment variable
(`MIX10_RUN`, `MIX10_EVAL`, `MIX10_RANDOM_EVAL`, `MIX10_CLEAN_RUN`, `MIX10_CLEAN_EVAL`, `MIX10_CLEAN_RANDOM_EVAL`;
`mix10_eval.sh` sets them when it executes the copy in the evaluation folder).

In [ ]:
import os

ATTACK = "datareplay"  # contaminating attack of this experiment
ATTACK_SPLIT = "test_datareplay"  # its test split
SEED = int(os.environ.get("MIX10_SEED", "0"))
RUN_ID = f"model-t32m10-{ATTACK}-seed{SEED}"
RUN = os.environ.get("MIX10_RUN") or f"src/runs/pretraining/all/T32_mix10_{ATTACK}/{RUN_ID}"
EVAL = os.environ.get("MIX10_EVAL") or f"src/runs/evaluation/all/T32_mix10_{ATTACK}/{RUN_ID}"
RANDOM_EVAL = os.environ.get("MIX10_RANDOM_EVAL") or f"{EVAL}-random-init"
CLEAN_RUN = os.environ.get("MIX10_CLEAN_RUN") or "src/runs/pretraining/all/T32_benign/model-t32b-seed0"
CLEAN_EVAL = os.environ.get("MIX10_CLEAN_EVAL") or "src/runs/evaluation/all/T32_benign/model-t32b-seed0"
CLEAN_RANDOM_EVAL = os.environ.get("MIX10_CLEAN_RANDOM_EVAL") or f"{CLEAN_EVAL}-random-init"

Setup: repo-root check (RULE 5), imports, the run and evaluation folders. Stops here with a clear message if the
experiment's run folder has no `history.jsonl` yet (not started); warns loudly if the run has not finished (no
`summary.json`, or a pause / budget stop) or is a smoke / short run.

In [ ]:
import json
from pathlib import Path

assert (
    Path("src/model/all_t32_benign/timesnet/plots.py").is_file() and Path("CLAUDE.md").is_file()
), "run this notebook from the repo root"

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.model.all_t32_benign.timesnet import plots
from src.model.all_t32_benign.timesnet.base_config import PretrainConfig
from src.model.all_t32_benign.timesnet.config import T32BenignConfig
from src.model.all_t32_benign.timesnet.data import PretrainFirewall, load_split
from src.model.all_t32_benign.timesnet.heads import PretrainingModel

EXPECTED_ENCODER_PARAMS = 2_301_312  # D12: d = 128, d_ff = 64 (architecture fact, not a result)
EXPECTED_TOTAL_PARAMS = 2_402_461  # encoder + recon / P1-P3 / projection heads + the 13-number mask token

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)


def show(fig):
    # Displays a figure returned by plots.py; None means the helper printed why it skipped.
    if fig is not None:
        plt.show()


FINISHED_REASONS = ("max_epochs", "early_stopping")


def eval_folder(path):
    # EvalFolder for a path with summary.json, else None (evaluation not run or not complete).
    p = Path(path)
    return plots.EvalFolder(p) if (p / "summary.json").is_file() else None


defaults = T32BenignConfig()
if not (Path(RUN) / "history.jsonl").is_file():
    raise FileNotFoundError(
        f"mix10_{ATTACK}: run folder {RUN} has no history.jsonl yet (training not started or another path). "
        f"Train first: bash src/model/all_t32_benign/experiments/mix10_train.sh {ATTACK}"
    )
run = plots.RunFolder(Path(RUN))
FINISHED = run.summary.get("stop_reason") in FINISHED_REASONS
ev = eval_folder(EVAL)
ev_random = eval_folder(RANDOM_EVAL)
if not FINISHED:
    print(
        f"*** RUN NOT FINISHED (summary stop_reason: {run.summary.get('stop_reason')!r}; epochs in history: "
        f"{int(run.epochs.epoch.max()) if not run.epochs.empty else 0}): partial curves only; the evaluation, if any, "
        "is of an intermediate best.pt ***"
    )
if f"T32_mix10_{ATTACK}" not in str(run.config.get("data_dir", "")):
    print(f"*** WARNING: the run's data_dir {run.config.get('data_dir')!r} is not T32_mix10_{ATTACK} ***")
DATA_DIR = Path(run.config.get("data_dir", defaults.data_dir))
CACHE_DIR = Path(run.config.get("cache_dir", defaults.cache_dir))
META = json.loads((DATA_DIR / "metadata.json").read_text())
FEATURES = META["features"]
print("run folder:       ", run.path)
print("evaluation folder:", ev.path if ev else f"none yet ({EVAL}; section 5 is skipped)")
print("random-init eval: ", ev_random.path if ev_random else f"none yet ({RANDOM_EVAL})")
print("finished:         ", FINISHED, "| stop_reason", run.summary.get("stop_reason"))
print("data / cache:     ", DATA_DIR, "/", CACHE_DIR)
if run.is_short():
    print("\n*** SHORT / SMOKE RUN: plumbing check of the notebook; the numbers below are not results ***")
if ev is not None and ev.smoke:
    print("*** SMOKE EVALUATION: a few windows per split; every test number below is plumbing only, not a result ***")

## 1. Input

Seven splits on disk. Train / val hold the benign windows of the `all/T32_benign` train / val vehicles **plus DataReplay
windows at 10% of the split** (the contamination; labels are never read here, so the mix share below comes from
`metadata.json`); test and the four `test_<attack>` splits are those of `all/T32_benign` (same windows, normalised with
the mixed-train statistics). Counts come from `metadata.json`, next to the window count of each numpy cache manifest.
Only train and val are opened before section 5.

In [ ]:
order = ["train", "val", "test", *plots.ATTACKS]
cbs = pd.DataFrame(META["counts_by_split"]).T
cbs = cbs.loc[[s for s in order if s in cbs.index]]
wanted = ["windows", "messages", "links", "vehicles", "padding_share", "windows_1_3", "share_1_3"]
cbs = cbs[[c for c in wanted if c in cbs.columns] + [c for c in cbs.columns if c not in wanted]]
for c in ("windows", "messages", "links", "vehicles", "windows_1_3"):
    if c in cbs:
        cbs[c] = pd.to_numeric(cbs[c], errors="coerce").astype("Int64")
manifest = {}
for split in cbs.index:
    path = CACHE_DIR / split / "manifest.json"
    manifest[split] = json.loads(path.read_text())["n_windows"] if path.is_file() else None
cbs["cache windows"] = pd.Series(manifest)
assert all(m is None or m == w for m, w in zip(cbs["cache windows"], cbs["windows"])), "cache differs from metadata"
display(cbs)
# contamination facts recorded by the input builder (any metadata key that mentions the mix), shown as stored
mix_keys = {k: v for k, v in META.items() if any(t in k.lower() for t in ("mix", "contamin", "attack_share"))}
if mix_keys:
    print(json.dumps(mix_keys, indent=2, default=str)[:4000])
else:
    plots.note("metadata.json has no mix / contamination key")

Load the two splits the run trained and selected on, through the pretraining firewall (train / val feature arrays
only; label files and test splits refused), and check them against `metadata.json`.

In [ ]:
firewall = PretrainFirewall()
train = load_split(CACHE_DIR, DATA_DIR, "train", firewall)
val = load_split(CACHE_DIR, DATA_DIR, "val", firewall)
for d in (train, val):
    assert len(d) == META["counts_by_split"][d.split]["windows"], d.split
assert firewall.label_files_opened == 0
print(
    f"train {len(train):,} / val {len(val):,} windows from {train.reader}; label files opened: "
    f"{firewall.label_files_opened}"
)
print(
    "run's env.json: rows",
    run.env.get("rows"),
    "| label_files_opened",
    run.env.get("label_files_opened"),
    "| train_rows_filter",
    run.env.get("train_rows_filter"),
)

Shape: each window is up to 32 messages (1 Hz) of one link × 13 features, padded to 32 rows with a prefix mask; the
encoder takes x (32 × 13) and the mask separately (masked FFT periods, masked pooling). Real rows per window:

In [ ]:
rows = []
lengths = {}
for d in (train, val):
    n = d.mask.sum(1).numpy()
    lengths[d.split] = n
    rows.append(
        {
            "split": d.split,
            "min": int(n.min()),
            "median": float(np.median(n)),
            "mean": round(float(n.mean()), 2),
            "max": int(n.max()),
            "share = 32": round(float((n == 32).mean()), 3),
            "share 1-3": round(float((n <= 3).mean()), 3),
            "share 1 (no recon target)": round(float((n < 2).mean()), 3),
            "padding share": round(1 - float(d.mask.float().mean()), 3),
        }
    )
display(pd.DataFrame(rows).set_index("split"))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(lengths["train"], bins=np.arange(1, 34) - 0.5, color="tab:blue")
ax.set_xlabel("real messages per window")
ax.set_ylabel("train windows (log)")
ax.set_yscale("log")
ax.set_title(f"Train (benign + 10% {ATTACK}): real messages per window")
plt.show()

Per-feature statistics over the **real** train rows, in the stored scaling (z-scores of the **mixed** train from
`metadata.json`, so ≈ 0 / 1). The trainer re-expresses `claimed_pos_x/y` and `range` with median / IQR and a soft
tail (D13); that scaling is shown in section 3.

In [ ]:
real = train.x[train.mask].numpy()
pd.DataFrame({"mean": real.mean(0), "std": real.std(0), "min": real.min(0), "max": real.max(0)}, index=FEATURES).round(
    3
)

One train window: the first full (32-row) window of the train split, as a heat map of its 13 features (stored scaling).

In [ ]:
i = int(np.flatnonzero(lengths["train"] == 32)[0])
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(train.x[i].numpy().T, aspect="auto", cmap="coolwarm", vmin=-3, vmax=3)
ax.set_yticks(range(len(FEATURES)), FEATURES)
ax.set_xlabel("message (time step, 1 Hz)")
ax.set_title(f"window {train.ids[i]}: {lengths['train'][i]} real rows", fontsize=9)
fig.colorbar(im, ax=ax, label="normalised value (train z-score)")
plt.show()
del real, train  # train is not needed again (the evaluation fitted Mahalanobis / kNN on it already)

## 2. Model

The run's settings rebuild `PretrainingModel` (this package's own `heads.py` + `encoder.py`): TimesNet
encoder (token embedding, 4 TimesBlocks, shared LayerNorm) + learned mask token + reconstruction head on H, P1–P3
heads and the SimCLR projection on z. Output shapes are for a batch of 2 full random windows; parameter counts are
asserted against D12 and the run's own preflight in `env.json`.

In [ ]:
model_settings = PretrainConfig.from_dict(run.config) if run.config else PretrainConfig.from_dict(vars(defaults))
torch.manual_seed(0)
model = PretrainingModel(model_settings).eval()
table = []
watch = {
    "encoder.embedding": model.encoder.embedding,
    **{f"encoder.blocks.{k}": b for k, b in enumerate(model.encoder.blocks)},
    "recon_head": model.recon_head,
    **{f"physics_heads.{k}": h for k, h in model.physics_heads.heads.items()},
    "proj_head": model.proj_head,
}
hooks = [
    m.register_forward_hook(
        lambda mod, inp, out, name=name: table.append(
            (name, type(mod).__name__, tuple(out.shape[1:]), sum(p.numel() for p in mod.parameters()))
        )
    )
    for name, m in watch.items()
]
x0 = torch.randn(2, model_settings.seq_len, model_settings.n_features)
m0 = torch.ones(2, model_settings.seq_len, dtype=torch.bool)
with torch.no_grad():
    h0, z0 = model.encoder(x0, m0)
    model.recon_head(h0)
    model.physics_heads(z0)
    model.proj_head(z0)
for h in hooks:
    h.remove()
table.insert(
    len(watch) - 5,
    (
        "encoder.norm (shared)",
        "LayerNorm",
        tuple(h0.shape[1:]),
        sum(p.numel() for p in model.encoder.norm.parameters()),
    ),
)
table.append(("mask_token", "Parameter", (model_settings.n_features,), model.mask_token.numel()))
display(pd.DataFrame(table, columns=["module", "type", "output (per window)", "params"]))

params_encoder = sum(p.numel() for p in model.encoder.parameters())
params_total = sum(p.numel() for p in model.parameters())
pre = run.env.get("preflight", {})
print(
    f"encoder {params_encoder:,} | total with heads + mask token {params_total:,} | H {tuple(h0.shape[1:])}, "
    f"z {tuple(z0.shape[1:])}"
)
print(
    "env.json preflight:",
    {k: pre.get(k) for k in ("params_encoder", "params_total")},
    "| summary.json:",
    {k: run.summary.get(k) for k in ("params_encoder", "params_total")},
)
assert params_encoder == EXPECTED_ENCODER_PARAMS and params_total == EXPECTED_TOTAL_PARAMS, (
    params_encoder,
    params_total,
)
assert (
    pre.get("params_encoder", params_encoder) == params_encoder
    and pre.get("params_total", params_total) == params_total
)

Objectives and their weights (D7: λ_recon = 1, λ_NT-Xent and λ_P1–P3 = 0.3; each raw loss is divided by a running
scale frozen after `norm_warmup_steps`), plus the run's preflight checks (every `padding_invariance_*` must be 0:
padding rows may not change any loss or z).

In [ ]:
keys = [
    "use_recon",
    "use_nce",
    "use_physics",
    "lambda_recon",
    "lambda_nce",
    "lambda_p1",
    "lambda_p2",
    "lambda_p3",
    "recon_ratio",
    "block_ratio",
    "temperature",
    "inject_prob",
    "min_rows_aux",
    "norm_warmup_steps",
    "robust_features",
    "soft_clip",
    "length_bucketed",
    "batch_size",
    "lr",
    "weight_decay",
    "warmup_steps",
    "grad_clip",
    "max_epochs",
    "max_steps",
    "patience",
    "seed",
    "min_train_rows",
    "short",
    "device",
]
display(
    pd.Series({k: run.config.get(k, "(not in config.json)") for k in keys}, name="config.json", dtype=object).to_frame()
)
if pre:
    display(pd.Series(pre, name="env.json preflight", dtype=object).to_frame())
    bad = {k: v for k, v in pre.items() if k.startswith("padding_invariance") and abs(v) > 1e-5}
    print("padding invariance:", "OK (all 0)" if not bad else f"NOT ZERO {bad}")
else:
    plots.note("env.json has no preflight block")

## 3. Training record

What the trainer wrote: `summary.json` (stop reason, best epoch by **val joint loss (mixed val)**, label-free), the
environment (device, GPU, label files opened = 0, the D13 scaling) and the first / last lines of `train.log`.

In [ ]:
if run.summary:
    print(json.dumps(run.summary, indent=2))
else:
    plots.note("no summary.json (run still going or stopped before writing it)")
env_keys = [
    "device",
    "cuda_device_name",
    "cuda_total_memory_gb",
    "torch",
    "python",
    "git_commit",
    "git_dirty",
    "label_files_opened",
    "label_paths_refused",
    "files_opened",
    "splits_opened",
    "load_seconds",
    "data_selection",
]
display(pd.Series({k: run.env.get(k) for k in env_keys if k in run.env}, name="env.json", dtype=object).to_frame())
scaling = run.env.get("scaling") or {}
if scaling:
    display(pd.DataFrame(scaling).T)
if run.log:
    print("\n".join(run.log if len(run.log) <= 12 else run.log[:4] + ["..."] + run.log[-6:]))

### 3a. Every loss, train vs val per epoch

Train = mean over the epoch's updates (a loss's mean uses only the updates where it was present: length-bucketed
batches of very short windows can lack NT-Xent / P-head losses); val = the mixed val check set (epoch 0 = random
init, no train value). Dotted line on P1–P3 = chance BCE (ln 2). Red = best epoch (lowest val joint).

In [ ]:
show(plots.TrainingPlots(run).losses())
cols = [
    c
    for c in [
        "epoch",
        "step",
        "lr",
        "train_joint",
        "val_joint",
        "train_recon",
        "val_recon",
        "val_recon_interp_baseline",
        "train_nce",
        "val_nce",
        "val_p1",
        "val_p2",
        "val_p3",
        "train_steps",
        "train_skipped",
        "grad_norm_mean",
        "grad_norm_max",
        "is_best",
        "bad_epochs",
        "partial",
    ]
    if c in run.history
]
if cols:
    display(run.history[cols].set_index("epoch").round(4))

### 3b. Reconstruction vs the linear-interpolation baseline (val)

The free baseline fills each hidden row by linear interpolation from its visible neighbours (same hidden rows). A
useful reconstruction must go **below** it; above it, the encoder is worse than a ruler. The baseline is computed
once on the first evaluation's seeded val masks (so it is flat) and pools over hidden rows, while `val_recon` averages
per-batch MSEs: close, not identical, definitions. The val check uses `eval_windows` val windows (summary.json).

In [ ]:
show(plots.TrainingPlots(run).recon_vs_interpolation())
h = run.history
if {"val_recon", "val_recon_interp_baseline"} <= set(h.columns) and run.best_epoch is not None:
    row = h[h.epoch == run.best_epoch].iloc[0]
    print(
        f"best epoch {run.best_epoch}: val recon {row.val_recon:.4f} vs interpolation {row.val_recon_interp_baseline:.4f}"
        f" (ratio {row.val_recon / row.val_recon_interp_baseline:.3f}; below 1 = better than interpolation)"
    )

### 3c. Per-update curves

From `steps.jsonl`: each loss per optimiser update (faint) and its rolling mean, the learning rate (warm-up then
cosine to its floor at the last planned update, X1) overlaid on the joint panel and alone, and the pre-clip gradient
norm with the clip value.

In [ ]:
show(plots.TrainingPlots(run).steps())
s = run.steps
if not s.empty:
    print(
        f"updates {len(s):,}; LR first / max / last {s.lr.iloc[0]:.3g} / {s.lr.max():.3g} / {s.lr.iloc[-1]:.3g}; "
        f"median s/update {s.step_seconds.median():.3f}"
        if "step_seconds" in s
        else ""
    )
    print("updates without a value, per loss:", {k: int(s[k].isna().sum()) for k in plots.LOSSES if k in s})

### 3d. Physics heads P1–P3: injected-violation detection on val

P1 (speed jump), P2 (speed scaled), P3 (sharp turn) are trained to detect violations **we inject** into benign windows
(D4). AUROC on val: 0.5 = guessing. This measures the pretext task, not attack detection.

In [ ]:
show(plots.TrainingPlots(run).physics_auroc())

### 3e. Representation health

Collapse monitors: effective rank of z (how many directions z uses; max d = 128), z std per dimension (mean / min),
alignment / uniformity, cosine of two views of the same window (`cos_pos`, should stay high) vs other windows
(`cos_neg`, should fall). Length shortcut: `length_probe_r2` (ridge R² of the real-row count from z; near 1 = length is
plainly readable from z) and `corr_znorm_length`. A high R² is why every test AUROC in section 5 sits next to the
length-only reference.

In [ ]:
show(plots.TrainingPlots(run).representation())

### 3f. Time and memory

Training and evaluation minutes per epoch (history), and the memory recorded at each evaluation (MPS driver / process
RSS on the Mac; CUDA peak allocated / reserved on EC2).

In [ ]:
show(plots.TrainingPlots(run).time_memory())

### 3g. Loss normalisation scales

Each raw loss is divided by a running scale (D7) that is frozen after the warm-up; the scales say how large each raw
loss was when they froze.

In [ ]:
show(plots.TrainingPlots(run).loss_scales())

## 4. Reconstruction on val

Per-feature reconstruction error on the hidden rows over epochs (`feature_history.jsonl`): train (pooled over every
hidden real row of the epoch's training batches, train mode), val, and the val interpolation baseline on the same
hidden rows. Values are in the trained scaling (D13). Red = best epoch.

In [ ]:
feature_plots = plots.FeaturePlots(run, FEATURES)
show(feature_plots.curves())

Per-feature val error at the best epoch next to the interpolation baseline (ratio < 1: the model beats interpolation
on that feature).

In [ ]:
fig, per_feature = feature_plots.best_bar()
show(fig)
if per_feature is not None:
    display(per_feature.sort_values("model (val)", ascending=False).round(4))

Three full (32-row) val windows (seeded pick) through `best.pt` on the CPU: the package's `ReconMasker` hides rows
(seeded), the encoder sees the mask token there, the reconstruction head predicts every row. Shaded = hidden rows;
crosses = linear interpolation of the hidden rows. Values in the trained scaling.

In [ ]:
best_pt = run.path / "best.pt"
if best_pt.is_file():
    examples = plots.ReconExamples(best_pt, META)
    full = np.flatnonzero(val.mask.sum(1).numpy() == 32)
    picks = np.sort(np.random.default_rng(0).choice(full, size=min(3, len(full)), replace=False))
    out = examples.run(val.x[picks], val.mask[picks], seed=0)
    print(f"checkpoint epoch {examples.epoch}, step {examples.step}; windows:", [val.ids[i] for i in picks])
    err = ((out["pred"] - out["x"]) ** 2 * out["hidden"][..., None]).sum((1, 2)) / (out["hidden"].sum(1) * 13)
    print("masked MSE of the hidden rows per window:", np.round(err, 4))
    show(
        plots.ReconExamples.plot(
            out, FEATURES, ["claimed_pos_x", "claimed_vel_x", "range"], [val.ids[i] for i in picks]
        )
    )
else:
    plots.note(f"{best_pt} missing")

## 5. Test

**Scores first, labels after.** `evaluate.py` scored every window with the frozen best checkpoint (masked
reconstruction error averaged over seeded mask passes; Mahalanobis on z fitted on benign train z; kNN cosine distance
to a benign train memory bank, K = 5 / 10 / 20 fixed in advance), saved `scores/<split>.npz`, and only then opened the
`_info` label shards (aligned by window id; every attack split must hold its own code only). Thresholds come from
the (mixed) val split. This notebook reads the saved scores and the summary; the class of a window is its split (benign `test`
vs `test_<attack>`), which evaluate's label check confirmed. Windows with < 2 real rows have nothing to hide: their
reconstruction score is the benign val median (filled, not measured).

In [ ]:
if ev is None:
    plots.note(f"no evaluation folder yet: run `bash src/model/all_t32_benign/experiments/mix10_eval.sh {ATTACK}`, then re-run")
else:
    checks = ev.label_checks()
    display(pd.Series(checks, name=ev.path.name, dtype=object).to_frame())
    assert checks["labels_read_after_scores"] is True, "labels were not read after the scores"
    if ev.smoke:
        print("*** SMOKE EVALUATION: plumbing only, not results ***")
    display(plots.TestPlots(ev).length_mix().round(4))
    print(
        "fits:",
        json.dumps(
            {k: ev.summary.get("fits", {}).get(k) for k in ("train_windows", "mahalanobis", "recon_undefined_fill")}
        ),
    )

AUROC per attack × score × subset (all windows, n ≥ 2 = reconstruction measured, n ≤ 3, n ≥ 4 real rows), each next
to the **length-only** reference on the same windows (score = − real rows, direction fixed in advance, F11), plus the
share of windows whose reconstruction score is filled. The `all` reconstruction AUROC depends on that fill; compare on
n ≥ 2 / n ≥ 4.

In [ ]:
if ev is not None:
    auroc = plots.AurocTable.wide(ev.summary)
    display(auroc.round(3))

Operating points at the benign val p95 threshold (recall of the attack windows, false-positive rate on the benign test
windows) and PR-AUC with its chance level, on all windows and on n ≥ 4; then per-track AUROC (track = one link, max /
mean of its window scores).

In [ ]:
if ev is not None:
    long = plots.AurocTable.long(ev.summary)
    ops = long[long.subset.isin(["all", "n_ge_4"])].set_index(["attack", "score", "subset"])
    display(ops[["AUROC", "PR-AUC", "chance", "recall@p95", "benign FPR@p95", "n benign", "n attack"]].round(3))
    display(plots.AurocTable.tracks(ev.summary).round(3))

Score histograms per window, benign test vs each attack, with the benign val p95 threshold (dashed). kNN is shown for
K = 10 (all three K are in the tables above).

In [ ]:
if ev is not None:
    show(plots.TestPlots(ev).histograms(("recon", "mahalanobis", "knn_k10")))

Measured reconstruction error (only windows with ≥ 2 real rows) against the real rows per window: median and IQR band
per length, benign test and each attack. DataReplay / DoS windows are 95–100% 1–3 rows (F11), so their curves rest on
few long windows.

In [ ]:
if ev is not None:
    show(plots.TestPlots(ev).recon_by_rows())

Random-init control: the same evaluation with an untrained, seed-fixed TimesNet and the run's scaling (`--random-init`).
An untrained encoder can already separate some attacks (kNN / Mahalanobis on random features, and length). Pretraining
adds what the difference shows, per subset.

In [ ]:
if ev is None:
    plots.note("no evaluation of the trained model")
elif ev_random is None:
    plots.note(f"no random-init evaluation folder ({RANDOM_EVAL}): run mix10_eval.sh {ATTACK}")
else:
    print("random-init folder:", ev_random.path, "| smoke:", ev_random.smoke)
    a = plots.AurocTable.long(ev.summary).set_index(["attack", "score", "subset"]).AUROC
    b = plots.AurocTable.long(ev_random.summary).set_index(["attack", "score", "subset"]).AUROC
    ctrl = pd.DataFrame({"trained": a, "random init": b, "trained - random": a - b}).reset_index()
    ctrl = ctrl[ctrl.subset.isin(["n_ge_2", "n_ge_4"]) & (ctrl.score != "length_only")]
    display(
        ctrl.pivot_table(
            index=["attack", "score"],
            columns="subset",
            values=["trained", "random init", "trained - random"],
            sort=False,
        ).round(3)
    )

## 6. Context (not a fair comparison)

The one-class ConvAE `task/autoencoder_benign_t32` (63,293 parameters, z = 16) used the **same split and the same test
windows**, but a different model, objective and scoring: it reconstructs every row without hiding any (so it also
scores 1-row windows), while TimesNet's reconstruction score is measured only on windows with ≥ 2 real rows and filled
below that. Its numbers are read from disk (the stored output of its notebook's results cell, else
`baselinepaper/01-our-studies/results.json`), never typed in. The like-for-like comparison is **GridSybil on n ≥ 4**
(the ConvAE's ">= 4 rows" AUROC); every other cell differs in subset or score definition. One seed each.

In [ ]:
baseline = plots.BaselineContext.load()
if baseline is None:
    plots.note("task/autoencoder_benign_t32 results not found on disk")
elif ev is None:
    plots.note("no TimesNet evaluation to put next to the baseline")
else:
    print("baseline source:", baseline.source)
    display(baseline.compare(ev.summary).round(3))

The numbers the analysis below refers to, computed from the files above (nothing typed in).

In [ ]:
key = {
    "run": run.run_id,
    "short run": run.is_short(),
    "evaluation": ev.path.name if ev else None,
    "smoke evaluation": ev.smoke if ev else None,
}
key.update(
    {
        k: run.summary.get(k)
        for k in (
            "best_epoch",
            "best_val_joint",
            "epochs_run",
            "steps",
            "stop_reason",
            "wall_seconds",
            "device",
            "params_encoder",
            "params_total",
        )
    }
)
h = run.history
if run.best_epoch is not None and not h.empty:
    row = h[h.epoch == run.best_epoch].iloc[0]
    for k in (
        "val_recon",
        "val_recon_interp_baseline",
        "auroc_p1",
        "auroc_p2",
        "auroc_p3",
        "z_effective_rank",
        "length_probe_r2",
        "cos_pos",
        "cos_neg",
    ):
        if k in row and pd.notna(row[k]):
            key[f"best epoch {k}"] = round(float(row[k]), 4)
if ev is not None:
    long = plots.AurocTable.long(ev.summary)
    for _, r in long[long.subset.isin(["all", "n_ge_2", "n_ge_4"])].iterrows():
        if r.AUROC is not None and pd.notna(r.AUROC):
            key[f"{r.attack}/{r.score}/{r.subset} AUROC"] = round(float(r.AUROC), 3)
key

## 7. Analysis

*Template: fill each line from the outputs above and section 8 after a full (non-smoke) run and evaluation. Do not keep
any line whose number was not computed here. One seed is not a result.*

- **Did pretraining work on the contaminated data?** Val joint (mixed val) from `<epoch 0>` to `<best>` at epoch
  `<best_epoch>` (stop `<stop_reason>`); recon / interpolation ratio `<ratio>`; P1–P3 val AUROC `<p1 / p2 / p3>`.
- **DataReplay (the contaminating attack):** AUROC n ≥ 2 / n ≥ 4 for recon / Mahalanobis / kNN k5 `<values>` vs clean
  `<values>` (Δ `<values>`, section 8); length-only `<values>`.
- **Other attacks:** Δ AUROC vs clean `<values>`; does contamination with DataReplay change them?
- **Pretrained vs random-init** for this run `<values>` and for the clean run `<values>`.
- **Operating point:** thresholds come from the mixed val (p95 of a val split that holds 10% DataReplay): recall / benign
  FPR at p95 `<values>` vs clean `<values>`.
- **Caveats:** one seed; F11 (DataReplay / DoS windows are 95–100% 1–3 rows, length separates them); F14; F1; the val
  numbers of this run and the clean run are on different val splits (mixed vs benign), so section 8 compares training
  curves only qualitatively; test numbers are on the same test windows.

## 8. Comparison with the clean run `model-t32b-seed0`

Same config, seed and test windows; the only difference is the training data (benign train vs benign + 10% DataReplay)
and, with it, the normalisation and the val split. Δ = mix10_datareplay − clean. Val-based numbers (val joint, val
recon, thresholds) are on **different val splits** (mixed vs benign): read them qualitatively. Test AUROCs are on the
same test windows and directly comparable. Skipped with a note when the clean run or an evaluation is missing.

In [ ]:
clean = plots.RunFolder(Path(CLEAN_RUN)) if (Path(CLEAN_RUN) / "history.jsonl").is_file() else None
clean_ev, clean_ev_random = eval_folder(CLEAN_EVAL), eval_folder(CLEAN_RANDOM_EVAL)
print("clean run:        ", clean.path if clean else f"missing ({CLEAN_RUN}; `npm run hf:download:t32b`)")
print("clean evaluation: ", clean_ev.path if clean_ev else f"missing ({CLEAN_EVAL})")
print("clean random-init:", clean_ev_random.path if clean_ev_random else f"missing ({CLEAN_RANDOM_EVAL})")

BEST_KEYS = ("val_joint", "val_recon", "val_recon_interp_baseline", "auroc_p1", "auroc_p2", "auroc_p3",
             "z_effective_rank", "length_probe_r2", "cos_pos", "cos_neg")
SUMMARY_KEYS = ("best_epoch", "best_val_joint", "epochs_run", "steps", "stop_reason", "wall_seconds",
                "train_windows", "val_windows", "final_lr", "device")


def run_facts(r):
    # summary keys + history values at the best epoch of a run (None where missing)
    out = {k: r.summary.get(k) for k in SUMMARY_KEYS}
    h = r.history
    if r.best_epoch is not None and not h.empty and "epoch" in h and (h.epoch == r.best_epoch).any():
        row = h[h.epoch == r.best_epoch].iloc[0]
        for k in BEST_KEYS:
            out[f"best epoch {k}"] = float(row[k]) if k in row and pd.notna(row[k]) else None
        if out.get("best epoch val_recon") and out.get("best epoch val_recon_interp_baseline"):
            out["best epoch recon / interpolation"] = (
                out["best epoch val_recon"] / out["best epoch val_recon_interp_baseline"]
            )
    return out


if clean is None:
    plots.note("clean run missing: section 8 skipped")
else:
    facts = pd.DataFrame({"clean": run_facts(clean), f"mix10_{ATTACK}": run_facts(run)})
    num = facts.apply(pd.to_numeric, errors="coerce")
    facts["Δ (mix − clean)"] = (num[f"mix10_{ATTACK}"] - num["clean"]).where(num.notna().all(axis=1))
    display(facts)

Val curves per epoch, clean vs mix (different val splits: shapes, not levels).

In [ ]:
if clean is not None:
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
    for ax, key, title in zip(
        axes,
        ("val_joint", "val_recon", "z_effective_rank"),
        ("val joint (own val split)", "val recon (own val split)", "z effective rank (own val split)"),
    ):
        for r, label, color in ((clean, "clean", "#2a78d6"), (run, f"mix10_{ATTACK}", "#eb6834")):
            y = plots.column(r.history, key)
            if y is not None:
                ax.plot(r.history.epoch, y, marker="o", ms=3, lw=2, color=color, label=label)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("epoch")
        ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

Per-feature val reconstruction error at each run's best epoch (own val split) and the ratio to interpolation.

In [ ]:
if clean is not None:
    _, f_clean = plots.FeaturePlots(clean, FEATURES).best_bar()
    _, f_mix = plots.FeaturePlots(run, FEATURES).best_bar()
    plt.close("all")
    if f_clean is not None and f_mix is not None:
        cols = [c for c in ("model (val)", "model / interpolation") if c in f_clean and c in f_mix]
        feat = pd.concat({"clean": f_clean[cols], f"mix10_{ATTACK}": f_mix[cols]}, axis=1)
        for c in cols:
            feat[("Δ", c)] = feat[(f"mix10_{ATTACK}", c)] - feat[("clean", c)]
        display(feat.round(4))

Test AUROC per attack × score, clean vs mix10_datareplay, on all / n ≥ 2 / n ≥ 4 real rows (window level). The row of
the contaminating attack (DataReplay) is the main question; the length-only reference is listed for each subset (it
differs between the runs only through the scoring of the same windows, so it should be equal).

In [ ]:
SCORES = ("recon", "mahalanobis", "knn_k5", "knn_k10", "knn_k20", "length_only")
SUBS = ("all", "n_ge_2", "n_ge_4")


def auroc_series(e):
    df = plots.AurocTable.long(e.summary)
    df = df[df.score.isin(SCORES) & df.subset.isin(SUBS)]
    return df.set_index(["attack", "score", "subset"]).AUROC.astype(float)


def side_by_side(a, b, la, lb):
    t = pd.DataFrame({la: auroc_series(a), lb: auroc_series(b)})
    t["Δ"] = t[lb] - t[la]
    return t.unstack("subset").reorder_levels([1, 0], axis=1).sort_index(axis=1, level=0, sort_remaining=False)


if clean is None or clean_ev is None or ev is None:
    plots.note("needs the clean evaluation and this run's evaluation")
else:
    cmp_trained = side_by_side(clean_ev, ev, "clean", f"mix10_{ATTACK}")
    display(cmp_trained.round(3))

The same for the random-init controls (untrained TimesNet with each run's scaling) and pretraining gain (trained − random-init) per run.

In [ ]:
if clean_ev_random is None or ev_random is None or clean_ev is None or ev is None:
    plots.note("needs both random-init evaluations and both trained evaluations")
else:
    display(side_by_side(clean_ev_random, ev_random, "clean random-init", f"mix10_{ATTACK} random-init").round(3))
    gain = pd.DataFrame(
        {
            "clean gain": auroc_series(clean_ev) - auroc_series(clean_ev_random),
            f"mix10_{ATTACK} gain": auroc_series(ev) - auroc_series(ev_random),
        }
    )
    gain = gain[gain.index.get_level_values("score") != "length_only"]
    display(gain.unstack("subset").round(3))

Δ AUROC (mix − clean) on n ≥ 2 for recon / Mahalanobis / kNN k5, per test attack; the contaminating attack is outlined.

In [ ]:
if clean_ev is not None and ev is not None:
    d = (auroc_series(ev) - auroc_series(clean_ev)).xs("n_ge_2", level="subset")
    attacks = list(plots.ATTACKS.values())
    scores = ("recon", "mahalanobis", "knn_k5")
    colors = ("#2a78d6", "#eb6834", "#1baf7a")
    fig, ax = plt.subplots(figsize=(9, 3.8))
    width = 0.26
    for j, (s, c) in enumerate(zip(scores, colors)):
        vals = [d.get((a, s), np.nan) for a in attacks]
        bars = ax.bar(np.arange(len(attacks)) + (j - 1) * width, vals, width - 0.02, color=c, label=s)
        for a, bar in zip(attacks, bars):
            if a == plots.ATTACKS[ATTACK_SPLIT]:
                bar.set_edgecolor("black")
                bar.set_linewidth(1.5)
    ax.axhline(0, color="grey", lw=1)
    ax.set_xticks(np.arange(len(attacks)), attacks)
    ax.set_ylabel("Δ AUROC (mix − clean), n ≥ 2")
    ax.set_title(f"Effect of 10% {plots.ATTACKS[ATTACK_SPLIT]} in train on test AUROC (window, n ≥ 2)")
    from matplotlib.patches import Patch

    ax.legend(handles=[Patch(color=c, label=s) for s, c in zip(scores, colors)], fontsize=8)
    fig.tight_layout()
    plt.show()

Operating point at the val p95 threshold (each run's own val: benign for clean, mixed for this run), on all windows.

In [ ]:
if clean_ev is not None and ev is not None:
    def ops(e):
        df = plots.AurocTable.long(e.summary)
        df = df[(df.subset == "all") & df.score.isin(("recon", "mahalanobis", "knn_k5"))]
        return df.set_index(["attack", "score"])[["recall@p95", "benign FPR@p95"]].astype(float)

    o = pd.concat({"clean": ops(clean_ev), f"mix10_{ATTACK}": ops(ev)}, axis=1)
    o[("Δ", "recall@p95")] = o[(f"mix10_{ATTACK}", "recall@p95")] - o[("clean", "recall@p95")]
    o[("Δ", "benign FPR@p95")] = o[(f"mix10_{ATTACK}", "benign FPR@p95")] - o[("clean", "benign FPR@p95")]
    display(o.round(3))